# 07 – File formats in the lake (Spark)

**Goal (Task 4, part 1):**
1. Start a **Spark session in local mode** with PySpark.
2. Read the **Parquet file of the full load** (notebook 01) from the lake.
3. Write the same data back to the lake as **CSV, JSON, Avro and Parquet**, with different **compression** and **partitioning** settings, and compare the **file sizes**.
4. Read every version back and compare the **content and schema (data types)**.
5. Look **inside** one Avro file and one Parquet file.

| Format | Stores data by | Schema inside the file? | Types | Human-readable |
|---|---|---|---|---|
| CSV | row (text) | no (only a header) | everything is text | yes |
| JSON | row (text) | no (field names in every row) | string / number / boolean | yes |
| Avro | **row** (binary) | **yes**, in the header | yes | no |
| Parquet | **column** (binary) | **yes**, in the footer | yes | no |

## Step 1 – Spark session (local mode)

`get_spark()` (in `spark_pile.py`) starts Spark with `master("local[*]")`: Spark runs **inside this container** and uses all CPU cores, with no cluster.
The first start downloads two extra packages from Maven (`spark-avro` for Avro, `hadoop-aws` for `s3a://` = Silo), which takes about a minute.

In [ ]:
import pandas as pd
from pyspark.sql import functions as F
from spark_pile import get_spark, list_files, copy_to_local, LAKE_URI

spark = get_spark("07-file-formats")
print("Spark", spark.version, "| master:", spark.sparkContext.master, "| lake:", LAKE_URI)
print("Spark UI: http://localhost:4040")

## Step 2 – Read the Parquet file of the full load

Notebook 01 (Task 2) wrote `postgres/animals/ingestion_date=<date>/<load_id>.<file_id>.parquet` into the lake bucket `pawhome-lake`.

In [ ]:
source_files = [name for name, size in list_files(spark, f"{LAKE_URI}/postgres/animals") if name.endswith(".parquet")]
source_path = f"{LAKE_URI}/postgres/animals/{source_files[-1]}"
print("Reading:", source_path)

animals = spark.read.parquet(source_path)
print("Rows:", animals.count(), "| partitions in Spark:", animals.rdd.getNumPartitions())
animals.printSchema()

## Step 3 – Write the data in different formats

Everything goes to `pawhome-lake/spark/formats/animals/<variant>/`. Spark always writes a **folder** with one or more `part-*` files (one per partition) and a `_SUCCESS` marker.

**Compression settings** (`option("compression", ...)`):
- CSV / JSON: `none`, `gzip`
- Avro: `uncompressed`, `snappy`, `deflate`
- Parquet: `none`, `snappy` (Spark's default), `gzip`, `zstd`

In [ ]:
OUT = f"{LAKE_URI}/spark/formats/animals"

FORMAT_VARIANTS = [
    # name               format     options
    ("csv",              "csv",     {"header": "true", "compression": "none"}),
    ("csv_gzip",         "csv",     {"header": "true", "compression": "gzip"}),
    ("json",             "json",    {"compression": "none"}),
    ("json_gzip",        "json",    {"compression": "gzip"}),
    ("avro",             "avro",    {"compression": "uncompressed"}),
    ("avro_snappy",      "avro",    {"compression": "snappy"}),
    ("avro_deflate",     "avro",    {"compression": "deflate"}),
    ("parquet",          "parquet", {"compression": "none"}),
    ("parquet_snappy",   "parquet", {"compression": "snappy"}),
    ("parquet_gzip",     "parquet", {"compression": "gzip"}),
    ("parquet_zstd",     "parquet", {"compression": "zstd"}),
]

for name, fmt, options in FORMAT_VARIANTS:
    animals.write.mode("overwrite").format(fmt).options(**options).save(f"{OUT}/{name}")
    print("written", name)

### Partitioning options

- `repartition(n)` → shuffles the rows into **n** Spark partitions → **n files**
- `coalesce(1)` → merges partitions **without a shuffle** → 1 file
- `partitionBy("species")` → one **folder per value**: `species=dog/`, `species=cat/` … (the column is moved into the folder name)
- `repartition("species")` + `partitionBy("species")` → exactly one file per folder

In [ ]:
PARTITION_VARIANTS = [
    # name                                      how the DataFrame is split      partitionBy
    ("parquet_coalesce_1",                      lambda d: d.coalesce(1),        None),
    ("parquet_repartition_4",                   lambda d: d.repartition(4),     None),
    ("parquet_partitionBy_species",             lambda d: d.repartition(3),     ["species"]),
    ("parquet_repartition_partitionBy_species", lambda d: d.repartition("species"), ["species"]),
]

for name, split, partition_cols in PARTITION_VARIANTS:
    writer = split(animals).write.mode("overwrite").option("compression", "snappy")
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.parquet(f"{OUT}/{name}")
    print("written", name)

## Step 4 – Check the results: files and sizes

In [ ]:
def data_files_of(variant):
    return [(n, s) for n, s in list_files(spark, f"{OUT}/{variant}")
            if not n.split("/")[-1].startswith((".", "_"))]          # skip _SUCCESS and .crc files

original_kb = sum(s for n, s in list_files(spark, source_path)) / 1024
rows = [{"variant": "original (dlt parquet)", "files": 1, "size_kb": round(original_kb, 1)}]
for name in [v[0] for v in FORMAT_VARIANTS + PARTITION_VARIANTS]:
    files = data_files_of(name)
    rows.append({"variant": name, "files": len(files), "size_kb": round(sum(s for _, s in files) / 1024, 1)})

sizes = pd.DataFrame(rows)
sizes["vs_csv"] = (sizes.size_kb / sizes.loc[sizes.variant == "csv", "size_kb"].iloc[0]).round(2)
sizes

In [ ]:
ax = sizes.set_index("variant").size_kb.plot.barh(figsize=(8, 6), title="Size of the same 3,000 animals (KB)")
ax.invert_yaxis(); ax.set_xlabel("KB");

In [ ]:
# What does partitionBy look like in the lake? One folder per species:
pd.DataFrame(data_files_of("parquet_repartition_partitionBy_species"), columns=["file", "bytes"])

**What to notice**
- Text formats (CSV, JSON) are the **biggest**; JSON is bigger than CSV because **every row repeats the field names**.
- **Compression** makes everything smaller; gzip/zstd/deflate compress more than snappy, but snappy is faster.
- **Parquet** is the smallest: values of one column are stored together, so they compress very well.
- `repartition(4)` → 4 files; `partitionBy("species")` → a folder per species. Too many small files is bad for performance.

## Step 5 – Read everything back and compare the schemas

CSV is read twice: without `inferSchema` (Spark sees only text) and with `inferSchema` (Spark guesses the types by scanning the data).

In [ ]:
def read_variant(name, fmt, infer=True):
    path = f"{OUT}/{name}"
    if fmt == "csv":
        return spark.read.option("header", True).option("inferSchema", infer).csv(path)
    return spark.read.format(fmt).load(path)

read_back = {
    "csv (text only)":   read_variant("csv", "csv", infer=False),
    "csv (inferSchema)": read_variant("csv", "csv"),
    "json":              read_variant("json", "json"),
    "avro":              read_variant("avro", "avro"),
    "parquet":           read_variant("parquet_snappy", "parquet"),
}

schema_table = pd.DataFrame(
    {"original": {f.name: f.dataType.simpleString() for f in animals.schema.fields},
     **{label: {f.name: f.dataType.simpleString() for f in df.schema.fields} for label, df in read_back.items()}}
).loc[animals.columns]

def highlight(row):
    return ["" if v == row["original"] else "background-color: #ffe0e0" for v in row]

schema_table.style.apply(highlight, axis=1)

**What to notice** (red = the type changed):
- **CSV** has no types at all → everything is `string`. With `inferSchema` Spark guesses: `decimal` becomes `double`, `id` becomes `int`.
- **JSON** knows only text, numbers and true/false → `date` and `timestamp` become **string**, `decimal` becomes `double`, `long` stays `bigint`. JSON also sorts the columns alphabetically.
- **Avro** and **Parquet** store the schema in the file → the types come back the same.

## Step 6 – Compare the content

Each version is cast back to the original types and joined with the original by `id`. Then, for every column, we count rows whose value is different.

In [ ]:
def mismatches(df):
    aligned = df.select([F.col(f.name).cast(f.dataType).alias(f.name) for f in animals.schema.fields])
    joined = animals.alias("o").join(aligned.alias("r"), "id", "full_outer")
    checks = [F.sum(F.when(~F.col(f"o.{c}").eqNullSafe(F.col(f"r.{c}")), 1).otherwise(0)).alias(c)
              for c in animals.columns if c != "id"]
    return joined.agg(*checks).first().asDict()

content = pd.DataFrame({label: mismatches(df) for label, df in read_back.items()})
content.loc["ROWS"] = {label: df.count() for label, df in read_back.items()}
content[content.sum(axis=1) > 0] if (content.drop("ROWS").sum(axis=1) > 0).any() else content

In [ ]:
# Why do timestamps differ in CSV / JSON? Spark writes them as TEXT with milliseconds only (.SSS):
(animals.select("id", F.col("_ingested_at").cast("string").alias("original (microseconds)"))
 .join(read_back["csv (text only)"].select(F.col("id").cast("long").alias("id"),
                                         F.col("_ingested_at").alias("as text in the CSV")), "id")
 .show(2, truncate=False))

# Why does inferSchema break _load_id? It LOOKS like a number, so Spark guessed double:
(animals.select("id", "_load_id")
 .join(read_back["csv (inferSchema)"].select(F.col("id").cast("long").alias("id"),
                                           F.col("_load_id").alias("inferSchema read it as")), "id")
 .show(2, truncate=False))

**What to notice**
- **Parquet** and **Avro**: 0 differences, the round trip is **lossless**.
- **CSV / JSON**: all rows are there, but
  - timestamps lose their **microseconds** (they are written as text with milliseconds),
  - with `inferSchema` Spark **guesses wrong**: `_load_id` (`1795788427.5373216`) looks like a number → `double` → digits are lost,
  - types have to be guessed or cast back by hand.
- That's why lakes store analytical data in **Parquet** (or Avro for row-by-row streaming), not in CSV/JSON.

## Step 7 – Inside an Avro file and a Parquet file

The code in `file_inspect.py` was generated with an AI assistant. It downloads one file from the lake and walks through its bytes.

In [ ]:
from file_inspect import inspect_avro, inspect_parquet

avro_file = [n for n, _ in data_files_of("avro_deflate")][0]
parquet_file = [n for n, _ in data_files_of("parquet_snappy")][0]
local_avro = copy_to_local(spark, f"{OUT}/avro_deflate/{avro_file}", "inspect_tmp")
local_parquet = copy_to_local(spark, f"{OUT}/parquet_snappy/{parquet_file}", "inspect_tmp")

### Avro: row-based

`[magic "Obj\x01"] [header: schema as JSON + codec] [sync marker] [block: n rows, size, rows…, sync marker] [block…]`

In [ ]:
avro = inspect_avro(local_avro)

In [ ]:
avro["schema_fields"]      # the schema stored in the file header

In [ ]:
avro["blocks"]             # the data blocks; every block ends with the same sync marker

In [ ]:
avro["first_records"]      # whole rows, one after another

### Parquet: column-based

`[magic "PAR1"] [row group: column chunk "id", column chunk "name", …] [footer: schema + statistics] [footer length] ["PAR1"]`

In [ ]:
parquet = inspect_parquet(local_parquet)

In [ ]:
parquet["schema"]          # physical type (how it is stored) + logical type (what it means)

In [ ]:
parquet["column_chunks"]   # one chunk per column: compression, encodings, sizes, min/max statistics

**Avro vs Parquet**

| | Avro | Parquet |
|---|---|---|
| layout | rows one after another, in blocks | each column stored separately (column chunks) |
| schema | JSON in the **header** (start of the file) | in the **footer** (end of the file) |
| statistics | none | min / max / null count per column chunk → whole chunks can be skipped |
| good for | writing row by row, streaming (Kafka) | analytics: read only the columns you need |